# TP2 — Classification: Model Comparison (Wine)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/L2Math/blob/main/session2/tp2_correction.ipynb)

## Objective

Apply the models from Session 2 to a **multiclass classification** problem using the Wine dataset (3 classes).

In Session 1 TP2, we learned classification metrics with a single model (Logistic Regression) on a binary problem. Here, we move to **model comparison and tuning** with five new algorithms.

### How does it differ from Session 1 TP2?

| | Session 1 TP2 | Session 2 TP2 |
|---|---|---|
| **Classes** | 2 (binary) | 3 (multiclass) |
| **Models** | Logistic Regression only | KNN, Tree, SVM, RF, GB |
| **Focus** | Learn classification metrics | Model comparison & tuning |

**Roadmap:** Quick EDA → Prepare (scaling demo!) → Baseline → Try 5 models → Compare → Tune with GridSearchCV → Cross-validate

> **Links:** [Session 1 TP2 — Binary Classification](../session1/tp2.ipynb) | [TP0 notebooks](.) for model theory reminders


---
## 1. Setup


In [ ]:
!pip install scikit-learn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay


---
## 2. Load & Explore the Data

The **Wine** dataset contains 178 samples of wine from three different cultivars grown in Italy. Each sample has 13 chemical measurements (alcohol, malic acid, ash, etc.). The task is to predict which cultivar (class 0, 1, or 2) a wine belongs to.


### Exercise 1.1 — Load the dataset

Load the Wine dataset and display its shape, target names, feature names, and first rows.

**Why:** Understand the data — how many samples, how many features, how many classes.

*Hint:* `load_wine(as_frame=True)`, `data.target_names`, `df.head()`


In [ ]:
data = load_wine(as_frame=True)
df = data.frame

print(f"Shape: {df.shape}")
print(f"\nTarget names: {data.target_names}")
print(f"\nFeatures: {list(data.feature_names)}")
df.head()


The Wine dataset has **178 samples**, **13 chemical features**, and **3 classes** (cultivar 0, 1, 2).

Features include alcohol content, malic acid, ash, alkalinity of ash, magnesium, total phenols, flavanoids, nonflavanoid phenols, proanthocyanins, color intensity, hue, OD280/OD315 of diluted wines, and proline.


### Exercise 1.2 — Summary statistics

Display summary statistics for the features.

**Why:** Check the range of each feature — are they on similar scales?

*Hint:* `df.describe()`

**Question:** Do features have similar scales?


In [ ]:
df.describe()


**Answer:** No! Features have very different scales. For example, `proline` ranges up to ~1600 while `nonflavanoid_phenols` ranges 0.1–0.7. This will be a problem for distance-based models (KNN, SVM) — we will need to scale the data.


### Exercise 1.3 — Check class balance

Count how many samples belong to each class.

**Why:** Imbalanced classes can mislead accuracy. We need to verify that classes are reasonably balanced before using accuracy as our main metric.

*Hint:* `y.value_counts()` and `y.value_counts(normalize=True)`


In [ ]:
y = df['target']

print("Class counts:")
print(y.value_counts())
print("\nClass proportions:")
print(y.value_counts(normalize=True))


Classes are reasonably balanced (~33% each). Accuracy is a meaningful metric here.


---
## 3. Prepare the Data


**Critical for this dataset:** Features have very different scales. This **dramatically affects distance-based models** (KNN, SVM). We will first see the effect **without** scaling, then **with** scaling. This is one of the most important lessons in this TP.


### Exercise 2.1 — Split the data (NO scaling yet)

Separate features and target, then split into train/test (80/20, `random_state=42`, `stratify=y`).

**Why:** Stratified split ensures each class is proportionally represented in both train and test sets — important for small multiclass datasets.

*Hint:* `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`


In [ ]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


### Exercise 2.2 — Quick test: KNN without scaling

Train a `KNeighborsClassifier(n_neighbors=5)` on the **unscaled** data and report accuracy.

**Why:** See how KNN performs when features have very different scales — the result may surprise you.

*Hint:* `.fit(X_train, y_train)`, `accuracy_score(y_test, model.predict(X_test))`


In [ ]:
knn_unscaled = KNeighborsClassifier(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)
acc_unscaled = accuracy_score(y_test, knn_unscaled.predict(X_test))
print(f"KNN accuracy WITHOUT scaling: {acc_unscaled:.4f}")


### Exercise 2.3 — Scale features and test KNN again

Apply `StandardScaler`, retrain KNN, and compare accuracy.

**Why:** See the dramatic improvement that scaling provides for distance-based models.

*Hint:* `scaler.fit_transform(X_train)`, `scaler.transform(X_test)` — fit on train only!


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNeighborsClassifier(n_neighbors=5)
knn_scaled.fit(X_train_scaled, y_train)
acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))
print(f"KNN accuracy WITH scaling: {acc_scaled:.4f}")
print(f"Improvement: {acc_scaled - acc_unscaled:+.4f}")


**Question:** Why does scaling improve KNN so dramatically?


**Answer:** KNN uses distances to find neighbors. Without scaling, features with large ranges (like `proline` ~0–1600) dominate the distance computation, making other features nearly irrelevant. After scaling, all features contribute equally to the distance.


**From now on, we use scaled data for all models.**


---
## 4. Baseline — Logistic Regression


### Exercise 3.1 — Train and evaluate baseline

Train a `LogisticRegression(max_iter=5000)` on the scaled data and display accuracy + classification report.

**Why:** Establish a baseline to beat. Logistic Regression is a simple, well-understood model.

*Hint:* `classification_report(y_test, y_pred, target_names=data.target_names)`


In [ ]:
baseline = LogisticRegression(max_iter=5000)
baseline.fit(X_train_scaled, y_train)

y_pred = baseline.predict(X_test_scaled)
baseline_acc = accuracy_score(y_test, y_pred)

print(f"=== Baseline: Logistic Regression ===")
print(f"Test Accuracy: {baseline_acc:.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=data.target_names)}")


Your goal: **beat this accuracy** with the models from Session 2.


---
## 5. Explore Different Models

Now we systematically try the five models from Session 2, each with a few hyperparameter settings. We track each model's best test accuracy to compare them at the end.


In [ ]:
results = {'Logistic Regression': baseline_acc}


### Exercise 4.1 — KNN Classifier

Try different values of `k` and report train/test accuracy for each. Track the best.

**Why:** The number of neighbors `k` controls the bias-variance trade-off. Small `k` → flexible (low bias, high variance); large `k` → smooth (high bias, low variance).

*Hint:* Loop over `k_values = [1, 3, 5, 10, 15]`, fit `KNeighborsClassifier(n_neighbors=k)`.

**Question:** Which `k` gives the best accuracy? What happens at `k=1` (train vs test)?


In [ ]:
print("=== KNN Classifier ===")
k_values = [1, 3, 5, 10, 15]
best_acc, best_k = 0, None

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, knn.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, knn.predict(X_test_scaled))
    print(f"  k={k:>2d}:  Train acc = {train_acc:.4f},  Test acc = {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc, best_k = test_acc, k

results['KNN'] = best_acc
print(f"\nBest: k={best_k}, Test accuracy = {best_acc:.4f}")


**Answer:** Small `k` overfits — `k=1` has perfect train accuracy but lower test accuracy. As `k` increases, the model smooths out. The best `k` balances flexibility and generalization.


### Exercise 4.2 — Decision Tree Classifier

Try different `max_depth` values and report train/test accuracy for each.

**Why:** Trees without depth limits grow until they memorize the training data (overfitting). `max_depth` controls complexity.

*Hint:* Loop over `depths = [2, 5, 10, None]`, fit `DecisionTreeClassifier(max_depth=depth, random_state=42)`.

**Question:** Compare train vs test accuracy for `max_depth=None`. What do you observe?


In [ ]:
print("=== Decision Tree Classifier ===")
depths = [2, 5, 10, None]
best_acc, best_depth = 0, None

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, tree.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, tree.predict(X_test_scaled))
    depth_str = str(depth) if depth is not None else "None"
    print(f"  max_depth={depth_str:>4s}:  Train acc = {train_acc:.4f},  Test acc = {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc, best_depth = test_acc, depth

results['Decision Tree'] = best_acc
print(f"\nBest: max_depth={best_depth}, Test accuracy = {best_acc:.4f}")


**Answer:** `max_depth=None` gives perfect train accuracy (1.0) but often lower test accuracy — clear overfitting. The tree has memorized the training data. Limiting depth forces the tree to learn more general patterns.


### Exercise 4.3 — SVM Classifier

Try different kernel/C combinations and report test accuracy.

**Why:** The kernel determines the shape of the decision boundary (linear vs non-linear). `C` controls the regularization strength (higher C = less regularization = tighter fit).

*Hint:* Loop over configs, fit `SVC(**cfg)`.

**Question:** Which kernel/C combination works best?


In [ ]:
print("=== SVM Classifier ===")
configs = [
    {'kernel': 'linear', 'C': 1.0},
    {'kernel': 'rbf', 'C': 0.1},
    {'kernel': 'rbf', 'C': 1.0},
    {'kernel': 'rbf', 'C': 10.0},
]
best_acc, best_cfg = 0, None

for cfg in configs:
    svm = SVC(**cfg)
    svm.fit(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, svm.predict(X_test_scaled))
    print(f"  {cfg}: Test acc = {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc, best_cfg = test_acc, cfg

results['SVM'] = best_acc
print(f"\nBest: {best_cfg}, Test accuracy = {best_acc:.4f}")


**Answer:** On this dataset with scaled features, both linear and RBF kernels perform well. The RBF kernel can capture non-linear boundaries, and `C` controls the margin trade-off. Higher `C` fits the training data more tightly.


### Exercise 4.4 — Random Forest Classifier

Try different numbers of trees and maximum depths. Report train/test accuracy.

**Why:** Random Forests combine many decision trees to reduce overfitting. More trees generally help, and `max_depth` controls individual tree complexity.

*Hint:* Loop over configs, fit `RandomForestClassifier(**cfg, random_state=42)`.


In [ ]:
print("=== Random Forest Classifier ===")
configs = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 200, 'max_depth': None},
]
best_acc, best_cfg = 0, None

for cfg in configs:
    rf = RandomForestClassifier(**cfg, random_state=42)
    rf.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, rf.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, rf.predict(X_test_scaled))
    print(f"  {cfg}: Train acc = {train_acc:.4f}, Test acc = {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc, best_cfg = test_acc, cfg

results['Random Forest'] = best_acc
print(f"\nBest: {best_cfg}, Test accuracy = {best_acc:.4f}")


### Exercise 4.5 — Gradient Boosting Classifier

Try different learning rates, numbers of estimators, and max depths. Report train/test accuracy.

**Why:** Gradient Boosting builds trees **sequentially**, each one correcting the errors of the previous. The learning rate controls how much each tree contributes.

*Hint:* Loop over configs, fit `GradientBoostingClassifier(**cfg, random_state=42)`.


In [ ]:
print("=== Gradient Boosting Classifier ===")
configs = [
    {'n_estimators': 50, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 3},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 5},
]
best_acc, best_cfg = 0, None

for cfg in configs:
    gb = GradientBoostingClassifier(**cfg, random_state=42)
    gb.fit(X_train_scaled, y_train)
    train_acc = accuracy_score(y_train, gb.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, gb.predict(X_test_scaled))
    print(f"  lr={cfg['learning_rate']}, n={cfg['n_estimators']}, d={cfg['max_depth']}: Train acc = {train_acc:.4f}, Test acc = {test_acc:.4f}")
    if test_acc > best_acc:
        best_acc, best_cfg = test_acc, cfg

results['Gradient Boosting'] = best_acc
print(f"\nBest: {best_cfg}, Test accuracy = {best_acc:.4f}")


---
## 6. Model Comparison


### Exercise 5.1 — Comparison table

Create a DataFrame from the `results` dictionary, sorted by accuracy (descending).

**Why:** A clear summary table makes it easy to compare all models at a glance.

*Hint:* `pd.DataFrame(...)`, `.sort_values(...)`


In [ ]:
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'Test Accuracy': list(results.values())
}).sort_values('Test Accuracy', ascending=False).reset_index(drop=True)

print(comparison.to_string(index=False))


### Exercise 5.2 — Visualization

Create a horizontal bar chart comparing model accuracies. Highlight the best model.

**Why:** Visual comparison is more intuitive than a table of numbers.

*Hint:* `plt.barh(...)`, use different colors for the best model.


In [ ]:
plt.figure(figsize=(10, 5))
colors = ['green' if i == 0 else 'steelblue' for i in range(len(comparison))]
plt.barh(comparison['Model'], comparison['Test Accuracy'], color=colors, edgecolor='k')
plt.xlabel('Test Accuracy')
plt.title('Model Comparison — Accuracy on Wine Dataset')
plt.xlim(0, 1.05)
for i, (_, row) in enumerate(comparison.iterrows()):
    plt.text(row['Test Accuracy'] + 0.005, i, f"{row['Test Accuracy']:.4f}", va='center')
plt.tight_layout()
plt.show()


**Question:** Which model performed best? Were you able to beat the logistic regression baseline?


**Answer:** With proper scaling, most models perform well on this dataset. SVM and KNN tend to perform very well thanks to the clean features and proper scaling. The key insight is that scaling was critical for unlocking the potential of distance-based models.


---
## 7. Hyperparameter Tuning with GridSearchCV

So far, we manually tried a few hyperparameter values. `GridSearchCV` automates this process: it tries **all combinations** in a parameter grid and uses **cross-validation** to find the best one.

This is more systematic and reliable than manual tuning.


### Exercise 6.1 — GridSearchCV on SVM

Run a grid search over kernel, C, and gamma for SVC. Use 5-fold cross-validation.

**Why:** Systematically find the best hyperparameters instead of guessing.

*Hint:* `GridSearchCV(SVC(), param_grid, cv=5, scoring='accuracy', n_jobs=-1)`


In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.01, 0.1],
}

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
)
grid_search.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")


### Exercise 6.2 — Evaluate the tuned model

Use the best model from GridSearchCV to predict on the test set. Display accuracy, classification report, and confusion matrix.

**Why:** Verify that the tuned model generalizes well to unseen data — not just cross-validation folds.

*Hint:* `grid_search.best_estimator_` gives the best model, already fitted.


In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test_scaled)

print("=== Tuned SVM ===")
print(f"Test Accuracy: {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"\n{classification_report(y_test, y_pred_tuned, target_names=data.target_names)}")

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_tuned, display_labels=data.target_names, ax=ax, cmap='Blues')
plt.title('Confusion Matrix — Tuned SVM')
plt.show()

print(f"\n=== Improvement over baseline ===")
print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"Tuned accuracy:    {accuracy_score(y_test, y_pred_tuned):.4f}")


---
## 8. Cross-Validation

A single train/test split can be lucky or unlucky. Cross-validation gives a more robust estimate of model performance by rotating the test set across multiple folds.


### Exercise 7.1 — 5-fold cross-validation on the tuned model

Run 5-fold cross-validation on the full dataset using the best model from GridSearchCV.

**Why:** Get a robust estimate of how the model will perform on new data.

*Hint:* `cross_val_score(model, scaler.transform(X), y, cv=5, scoring='accuracy')`

**Question:** Is the model stable across folds (low standard deviation)?


In [ ]:
cv_scores = cross_val_score(
    grid_search.best_estimator_,
    scaler.transform(X), y,
    cv=5, scoring='accuracy'
)

print(f"5-Fold CV accuracy scores: {cv_scores}")
print(f"Mean accuracy: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")


**Answer:** The standard deviation is small, indicating the model performs consistently across different data splits. The model generalizes well and is not overly sensitive to the particular train/test partition.


---
## 9. Summary

### Key Takeaways

1. **Feature scaling is CRITICAL** for distance-based models — KNN went from ~70% to ~97% just by scaling!
2. **Multiple models should be compared** systematically on the same data split
3. **GridSearchCV** automates hyperparameter search with cross-validation
4. **Simple models** (Logistic Regression) are strong baselines — always start there
5. The **same methodology** applies to both regression (TP1) and classification (TP2)

### Session 2 Methodology

| Step | What you did |
|------|-------------|
| **EDA** | Quick exploration of features and target |
| **Scaling demo** | KNN without vs with scaling |
| **Baseline** | Logistic Regression |
| **Manual tuning** | 5 models with different hyperparameter settings |
| **Systematic tuning** | GridSearchCV on SVM |
| **Validation** | Cross-validation for robust performance estimate |

### What's Next?

You now have a complete workflow for classification problems. The same approach works for any dataset:

1. Explore and understand your data
2. Preprocess (scale if needed)
3. Establish a baseline
4. Try multiple models
5. Tune the best candidates
6. Validate with cross-validation
